In [ ]:
# This file introduces an improved implementation of Variant2 of Algorithm 13,
# which is faster than the old version.
# To reproduce the results, please use the HIGH-RAM CPU of google colab,
# which has 8 CPU cores and 50.99 GB of RAM.
# The improvement is partly accomplished by ChatGPT.


In [1]:
!apt-get update
!apt-get install -y libtbb-dev

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,061 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,945 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,347 kB]
Hit:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu

In [2]:
%%writefile Variant1_new_implementation.cpp

#include <iostream>
#include <vector>
#include <thread>
#include <mutex>
#include <algorithm>
#include <queue>
#include <random>
#include <stack>
#include <chrono>
#include <limits>
#include <tuple>
#include <iomanip>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;

const double INF = numeric_limits<double>::infinity();

// ============================================================
// GRAPH
// ============================================================

struct AdjEdge {
    int to;
    int id;
};

// ============================================================
// PRIM MST
// ============================================================

vector<int> primMST(const Matrix& dist) {

    int n = dist.size();

    vector<double> key(n, INF);
    vector<int> parent(n, -1);
    vector<char> inMST(n, 0);

    priority_queue<
        pair<double,int>,
        vector<pair<double,int>>,
        greater<>
    > pq;

    key[0] = 0.0;

    pq.emplace(0.0, 0);

    while (!pq.empty()) {

        auto [k, u] = pq.top();
        pq.pop();

        if (inMST[u])
            continue;

        inMST[u] = 1;

        const double* row = dist[u].data();

        for (int v = 0; v < n; ++v) {

            double w = row[v];

            if (w && !inMST[v] && w < key[v]) {

                key[v] = w;
                parent[v] = u;

                pq.emplace(w, v);
            }
        }
    }

    return parent;
}

// ============================================================
// BUILD GRAPH
// ============================================================

void buildGraph(
    int n,
    const vector<Edge>& edge_list,
    vector<vector<AdjEdge>>& graph)
{
    graph.assign(n, {});

    for (int i = 0; i < (int)edge_list.size(); ++i) {

        auto [u, v, w] = edge_list[i];

        graph[u].push_back({v, i});
        graph[v].push_back({u, i});
    }
}

// ============================================================
// FAST DFS
// ============================================================

inline void dfs_fast(
    int start,
    const vector<vector<AdjEdge>>& graph,
    const vector<char>& active,
    vector<int>& visited,
    int token,
    vector<int>& nodes)
{
    nodes.clear();

    stack<int> st;

    st.push(start);

    visited[start] = token;

    while (!st.empty()) {

        int u = st.top();
        st.pop();

        nodes.push_back(u);

        for (const auto& e : graph[u]) {

            if (!active[e.id])
                continue;

            int v = e.to;

            if (visited[v] != token) {

                visited[v] = token;

                st.push(v);
            }
        }
    }
}

// ============================================================
// MAIN THREAD
// ============================================================

void main_thread_func(
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int,int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    deque<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    // initially all active
    vector<char> active(num_edges, 1);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;

    while (true) {

        int task;

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.front();
            task_queue.pop_front();
        }

        // remove edge task
        active[task] = 0;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// WORKER
// ============================================================

void worker(
    int tid,
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int,int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    deque<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    vector<char> active(num_edges);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;

    int current_added = num_edges;

    int task;

    fill(active.begin(), active.end(), 0);

    while (true) {

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.back();
            task_queue.pop_back();
        }

        if (task < num_edges - 1){
        for (int i = current_added - 1; i >= task + 1; --i) {
            active[i] = 1;
        }
        }
        current_added = task + 1;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// MMJ
// ============================================================

Matrix cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
    const Matrix& distance_matrix,
    int n_jobs)
{


    int n = distance_matrix.size();

    Matrix mmj_matrix(
        n,
        vector<double>(n, 0.0));

    // ========================================================
    // TASK QUEUE
    // ========================================================

    deque<int> task_queue;

    for (int i = 0; i < n - 1; ++i)
        task_queue.push_back(i);

    mutex queue_mutex;

    // ========================================================
    // MST
    // ========================================================

    auto parent = primMST(distance_matrix);

    vector<Edge> edge_list;

    edge_list.reserve(n - 1);

    for (int i = 1; i < n; ++i) {

        edge_list.emplace_back(
            min(i, parent[i]),
            max(i, parent[i]),
            distance_matrix[i][parent[i]]);
    }

    sort(
        edge_list.begin(),
        edge_list.end(),
        [](const Edge& a, const Edge& b) {

            return get<2>(a) > get<2>(b);
        });

    // ========================================================
    // EDGE ARRAYS
    // ========================================================

    vector<pair<int,int>> edge_nodes;

    vector<double> edge_weights;

    edge_nodes.reserve(n - 1);
    edge_weights.reserve(n - 1);

    for (const auto& [u, v, w] : edge_list) {

        edge_nodes.emplace_back(u, v);

        edge_weights.push_back(w);
    }

    // ========================================================
    // GRAPH
    // ========================================================

    vector<vector<AdjEdge>> graph;

    buildGraph(n, edge_list, graph);

    // ========================================================
    // THREADS
    // ========================================================

    vector<thread> threads;

    for (int t = 0; t < n_jobs; ++t) {

        threads.emplace_back(
            worker,
            t,
            cref(graph),
            cref(edge_nodes),
            cref(edge_weights),
            ref(mmj_matrix),
            ref(task_queue),
            ref(queue_mutex));
    }

    // ========================================================
    // TIMING
    // ========================================================



    main_thread_func(
        graph,
        edge_nodes,
        edge_weights,
        mmj_matrix,
        task_queue,
        queue_mutex);

    for (auto& th : threads)
        th.join();



    return mmj_matrix;
}

// ============================================================
// DISTANCE MATRIX
// ============================================================

vector<vector<double>> createDistanceMatrix(
    int N,
    int seed)
{
    mt19937 gen(seed);

    uniform_real_distribution<double>
        dist(1.0, 19999.0);

    vector<vector<double>> A(
        N,
        vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {

        for (int j = i + 1; j < N; ++j) {

            double val =
                round(dist(gen) * 100.0) / 100.0;

            A[i][j] = val;
            A[j][i] = val;
        }
    }

    return A;
}


// ============================================================
// MAIN
// ============================================================

int main() {

    int N =49999;

    int n_jobs =
        thread::hardware_concurrency();

    int random_seed = 7875;

    cout << "Number of nodes: "
         << N << endl;

    cout << "Number of CPU cores: "
         << n_jobs << endl;

    auto distanceMatrix = createDistanceMatrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();

    auto mmjMatrix =
        cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
            distanceMatrix,
            n_jobs);

    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);

    cout << "Time used for MMJ matrix (new implementation of Variant1 of Algorithm 13):"
         << time_used
         << " seconds\n";

    cout << "Print last 30 values of the first row of mmj matrix:\n";

    const auto& row = mmjMatrix[0];

    for (size_t i = row.size() - 30; i < row.size(); ++i)
    {
        cout << fixed
             << setprecision(2)
             << row[i]
             << " ";
    }

    cout << "\n";

    return 0;
}



Writing Variant1_new_implementation.cpp


In [3]:
%%writefile Variant2_new_implementation.cpp

#include <iostream>
#include <vector>
#include <thread>
#include <mutex>
#include <algorithm>
#include <queue>
#include <random>
#include <chrono>
#include <limits>
#include <tuple>
#include <iomanip>
#include <cstdint>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;

const double INF = numeric_limits<double>::infinity();

struct AdjEdge {
    int to;
    int id;
};

// ============================================================
// PRIM MST
// ============================================================

vector<int> primMST(const Matrix& dist) {

    int n = dist.size();

    vector<double> key(n, INF);
    vector<int> parent(n, -1);
    vector<char> inMST(n, 0);

    priority_queue<
        pair<double, int>,
        vector<pair<double, int>>,
        greater<>
    > pq;

    key[0] = 0.0;

    pq.emplace(0.0, 0);

    while (!pq.empty()) {

        auto [k, u] = pq.top();
        pq.pop();

        if (inMST[u])
            continue;

        inMST[u] = 1;

        const double* row = dist[u].data();

        for (int v = 0; v < n; ++v) {

            double w = row[v];

            if (w && !inMST[v] && w < key[v]) {

                key[v] = w;
                parent[v] = u;

                pq.emplace(w, v);
            }
        }
    }

    return parent;
}

// ============================================================
// BUILD GRAPH
// ============================================================

void buildGraph(
    int n,
    const vector<Edge>& edge_list,
    vector<vector<AdjEdge>>& graph)
{
    graph.assign(n, {});

    for (int i = 0; i < (int)edge_list.size(); ++i) {

        auto [u, v, w] = edge_list[i];

        graph[u].push_back({v, i});
        graph[v].push_back({u, i});
    }
}

// ============================================================
// FAST DFS
// ============================================================

inline void dfs_fast(
    int start,
    const vector<vector<AdjEdge>>& graph,
    const vector<char>& active,
    vector<uint32_t>& visited,
    uint32_t token,
    vector<int>& nodes,
    vector<int>& stack_buffer)
{
    nodes.clear();

    stack_buffer.clear();

    stack_buffer.push_back(start);

    visited[start] = token;

    while (!stack_buffer.empty()) {

        int u = stack_buffer.back();
        stack_buffer.pop_back();

        nodes.push_back(u);

        const auto& neighbors = graph[u];

        for (const auto& e : neighbors) {

            if (!active[e.id])
                continue;

            int v = e.to;

            if (visited[v] != token) {

                visited[v] = token;

                stack_buffer.push_back(v);
            }
        }
    }
}

// ============================================================
// WORKER
// ============================================================

void worker(
    int tid,
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int, int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    queue<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    // active edges
    vector<char> active(num_edges, 1);

    // IMPORTANT:
    // each worker must progressively remove edges
    int current_removed = -1;

    vector<uint32_t> visited(n, 0);

    uint32_t token = 1;

    vector<int> tree1;
    vector<int> tree2;
    vector<int> stack_buffer;

    tree1.reserve(n);
    tree2.reserve(n);
    stack_buffer.reserve(n);

    while (true) {

        int task;

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.front();
            task_queue.pop();
        }

        // ====================================================
        // cumulative edge removals
        // ====================================================

        for (int i = current_removed + 1; i <= task; ++i)
            active[i] = 0;

        current_removed = task;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        // ====================================================
        // DFS 1
        // ====================================================

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1,
            stack_buffer
        );

        // ====================================================
        // DFS 2
        // ====================================================

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2,
            stack_buffer
        );

        // ====================================================
        // fill MMJ matrix
        // ====================================================

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// MAIN MMJ
// ============================================================

Matrix cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
    const Matrix& distance_matrix,
    int n_jobs)
{
    int n = distance_matrix.size();

    Matrix mmj_matrix(
        n,
        vector<double>(n, 0.0));

    queue<int> task_queue;

    mutex queue_mutex;

    for (int i = 0; i < n - 1; ++i)
        task_queue.push(i);

    // ========================================================
    // MST
    // ========================================================

    auto parent = primMST(distance_matrix);

    // ========================================================
    // EDGE LIST
    // ========================================================

    vector<Edge> edge_list;

    edge_list.reserve(n - 1);

    for (int i = 1; i < n; ++i) {

        int p = parent[i];

        edge_list.emplace_back(
            min(i, p),
            max(i, p),
            distance_matrix[i][p]
        );
    }

    // descending order
    sort(edge_list.begin(),
         edge_list.end(),
         [](const Edge& a, const Edge& b) {
             return get<2>(a) > get<2>(b);
         });

    // ========================================================
    // EDGE ARRAYS
    // ========================================================

    vector<pair<int, int>> edge_nodes;
    vector<double> edge_weights;

    edge_nodes.reserve(n - 1);
    edge_weights.reserve(n - 1);

    for (const auto& [u, v, w] : edge_list) {

        edge_nodes.emplace_back(u, v);
        edge_weights.push_back(w);
    }

    // ========================================================
    // GRAPH
    // IMPORTANT:
    // must build AFTER sorting edge_list
    // so edge IDs match task IDs
    // ========================================================

    vector<vector<AdjEdge>> graph;

    buildGraph(n, edge_list, graph);

    // ========================================================
    // THREADS
    // ========================================================

    vector<thread> threads;

    for (int t = 0; t < n_jobs; ++t) {

        threads.emplace_back(
            worker,
            t,
            cref(graph),
            cref(edge_nodes),
            cref(edge_weights),
            ref(mmj_matrix),
            ref(task_queue),
            ref(queue_mutex)
        );
    }

    for (auto& th : threads)
        th.join();

    return mmj_matrix;
}

// ============================================================
// DISTANCE MATRIX
// ============================================================

Matrix createDistanceMatrix(
    int N,
    int seed)
{
    mt19937 gen(seed);

    uniform_real_distribution<double>
        dist(1.0, 19999.0);

    Matrix A(
        N,
        vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {

        for (int j = i + 1; j < N; ++j) {

            double val =
                round(dist(gen) * 100.0) / 100.0;

            A[i][j] = val;
            A[j][i] = val;
        }
    }

    return A;
}

// ============================================================
// MAIN
// ============================================================

int main() {

    // WARNING:
    // Dense matrices explode in memory quickly

    int N = 49999;

    int n_jobs =
        thread::hardware_concurrency();

    int random_seed = 7875;

    cout << "Number of nodes: "
         << N << endl;

    cout << "Number of CPU cores: "
         << n_jobs << endl;

    auto distanceMatrix =
        createDistanceMatrix(
            N,
            random_seed
        );

    auto start = chrono::high_resolution_clock::now();

    auto mmjMatrix =
        cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
            distanceMatrix,
            n_jobs
        );

    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed
         << setprecision(3);

    cout << "Time used for MMJ matrix (new implementation of Variant2 of Algorithm 13):"
         << time_used
         << " seconds\n";

    cout << "Print last 30 values of first row of mmj matrix:\n";

    const auto& row = mmjMatrix[0];

    for (size_t i = row.size() - 30;
         i < row.size();
         ++i)
    {
        cout << fixed
             << setprecision(2)
             << row[i]
             << " ";
    }

    cout << "\n";

    return 0;
}


Writing Variant2_new_implementation.cpp


In [4]:
!g++ -std=c++17 -O3 -march=native Variant1_new_implementation.cpp  -o tt -ltbb
!./tt

Number of nodes: 49999
Number of CPU cores: 8
Time used for MMJ matrix (new implementation of Variant1 of Algorithm 13):26.126 seconds
Print last 30 values of the first row of mmj matrix:
2.04 1.58 1.58 1.66 1.56 1.56 1.75 1.56 1.56 2.03 1.83 1.71 1.58 1.56 2.20 1.56 1.56 1.93 1.56 1.65 1.80 1.56 2.29 1.56 1.56 1.56 1.60 1.78 1.62 1.56 


In [5]:
!g++ -std=c++17 -O3 -march=native Variant2_new_implementation.cpp  -o tt -ltbb
!./tt

Number of nodes: 49999
Number of CPU cores: 8
Time used for MMJ matrix (new implementation of Variant2 of Algorithm 13):24.990 seconds
Print last 30 values of first row of mmj matrix:
2.04 1.58 1.58 1.66 1.56 1.56 1.75 1.56 1.56 2.03 1.83 1.71 1.58 1.56 2.20 1.56 1.56 1.93 1.56 1.65 1.80 1.56 2.29 1.56 1.56 1.56 1.60 1.78 1.62 1.56 


In [ ]:

# I can be seen that after some tricks like faster Depth First Search (or DFS),
# Variant1 and Variant2 of Algorithm 13 perform similarly.

In [6]:
import platform
import psutil

# CPU information
print("CPU Information:")
print(f"Processor: {platform.processor()}")
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"CPU frequency: {psutil.cpu_freq().current:.2f} MHz")

# RAM information
ram = psutil.virtual_memory()

print("\nRAM Information:")
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")

CPU Information:
Processor: x86_64
Physical cores: 4
Logical cores: 8
CPU frequency: 2250.00 MHz

RAM Information:
Total RAM: 50.99 GB
